In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import xarray as xr
import os
from datetime import datetime
import matplotlib.pyplot as plt
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

current_path = Path.cwd()

# Search for the utilities directory
for parent in current_path.parents:
    candidate = parent / "utilities"
    if candidate.is_dir():
        sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate 'utilities' directory in parent paths.")

from utilities import init_notebook_environment
from utilities import get_dataset_products,get_prod_files,dataset_defaults,file_parser, get_dataset_dirs, get_dataset_products,run_globcolour_dataset
from utilities import get_nc_prod,product_defaults,netcdf_product_defaults,get_datasets_source,get_dataset_dirs
from utilities import run_stats_pipeline,build_stats_map, run_pp_pipeline, load_dataset
from utilities import build_product_attributes, run_psc_pipeline, find_text, dataset_defaults, parse_dataset_info, hash_grid
from utilities import get_file_dates, get_filepath, get_period_info, get_period_sets, get_period_dates, get_daterange, generate_periods
from utilities import get_clim_period,extract_period_code
from utilities import build_anomaly_map, run_anomaly_pipeline, batch_anomaly_match


env = init_notebook_environment(verbose=False)


In [8]:
prod='CHL'
base_prod_attrs = build_product_attributes(prod, is_anomaly=True)
base_prod_attrs.get('standard_name', f"{prod} Anomaly")

'mass_concentration_of_chlorophyll_in_sea_water_anomaly'

In [3]:
xr.open_dataset('/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL/M_199711-OCCCI-CHL-NES-ANOM.nc')

<xarray.Dataset> Size: 545kB
Dimensions:   (time: 1, lat: 312, lon: 432)
Coordinates:
  * time      (time) datetime64[ns] 8B 1997-11-01
  * lat       (lat) float64 2kB 46.98 46.94 46.9 46.85 ... 34.1 34.06 34.02
  * lon       (lon) float64 3kB -78.98 -78.94 -78.9 ... -61.1 -61.06 -61.02
Data variables:
    CHL_anom  (time, lat, lon) float32 539kB ...
Attributes: (12/39)
    conventions:                   CF-1.11, COARDS, ACDD-1.3
    acknowledgement:               The data are sponsored by NOAA and may be ...
    institution:                   DOC | NOAA | National Marine Fisheries Ser...
    naming_authority:              gov.noaa.nefsc
    license:                       The data may be used and redistributed for...
    creator_type:                  group
    ...                            ...
    time_coverage_duration:        P1D
    time_coverage_resolution:      P1D
    product_name:                  Concentration of chlorophyll-a anomaly
    anomaly_method:                ratio (stat / clim)
    source_files:                  M_199711-OCCCI-CHL-NES-STATS.nc, MONTH_11_...
    history:                       2026-10-14T13:14:00Z Computed CHL anomalie...

In [2]:
run_anomaly_pipeline('CHL', periods='M',map_subset='NES',verbose=True)


🚀 Starting anomaly pipeline for CHL | Period: M
🔍 Resolving output directory for CHL (M Anomalies)...
Searching for input M STATS files for None:CHL
Found 8 for input M STATS files for None:CHL
Searching for MONTH CLIM files for None:CHL
Found 12 for climatology M CLIM files for None:CHL
------------------------------------------------------------
📊 ANOMALY SUMMARY: CHL | M | NES
✅ Up-to-date:  0/8
⏳ To process:  8 files
------------------------------------------------------------
  📖 Loading climatology baseline into RAM: MONTH_11_1997_2020-OCCCI-CHL-NES-CLIM.nc
    ✅ Saved: M_199711-OCCCI-CHL-NES-ANOM.nc
    ✅ Saved: M_199811-OCCCI-CHL-NES-ANOM.nc
    ✅ Saved: M_199911-OCCCI-CHL-NES-ANOM.nc
  📖 Loading climatology baseline into RAM: MONTH_05_1997_2020-OCCCI-CHL-NES-CLIM.nc
    ✅ Saved: M_199805-OCCCI-CHL-NES-ANOM.nc
    ✅ Saved: M_200005-OCCCI-CHL-NES-ANOM.nc
  📖 Loading climatology baseline into RAM: MONTH_07_1997_2020-OCCCI-CHL-NES-CLIM.nc
    ✅ Saved: M_199807-OCCCI-CHL-NES-ANOM.

In [ ]:
stat_files = get_prod_files('CHL',period='M',map_subset='NES')
clim_files = get_prod_files('CHL',period='MONTH',map_subset='NES')
amap = batch_anomaly_match(stat_files,clim_files)
# 2. Loop through the successful matches to build your maps
for input_file, matched_clim in amap.items():
    print(f"Processing: {input_file} against {matched_clim}")



In [ ]:
ds = xr.open_mfdataset(stat_files)
print(list(ds.data_vars))

In [ ]:
build_anomaly_map('CHL',period='M',map_subset='NES',verbose=True)

In [ ]:
f = get_prod_files('SST',period='M',map_subset='NWA',verbose=False)
file_parser(f[0])

In [ ]:
files = get_prod_files('CHL',period='WEEK',map_subset='NES',verbose=False,data_type='STATS')
search_text = "STATS"
replace_text = "CLIM"

for file_path in files:
    path_obj = Path(file_path)
    
    # Check if the search string is in the actual file name (ignoring folders)
    if search_text in path_obj.name:
        # Replace the text in the file name
        new_name = path_obj.name.replace(search_text, replace_text)
        
        # Create the new full path
        new_path = path_obj.with_name(new_name)
        
        # Rename the file on the filesystem
        path_obj.rename(new_path)
        print(f"Renamed: {path_obj.name} -> {new_name}")

In [ ]:
get_filepath('CHL',period='M',verbose=False,map_subset='NES') 

In [ ]:
run_pp_pipeline(sst_dataset='CORALSST',map_subset='NWA',verbose=True)

In [ ]:
run_globcolour_dataset(subset_map="NWA", dry_run=False, overwrite=True, verbose=True, debug=False)